# HMDB51 zero-shot action recognition — CLIP4ClipCompressed and its ablations

Every HMDB51 class becomes a text prompt, and each video is assigned the class whose prompt embedding is closest to its video embedding. The similarity is the one the retrieval model was trained with. Nothing is fine-tuned on HMDB51, so this measures how well the MSR-VTT-trained text–video space transfers to action recognition, and how much each compressed-domain branch (I-frame, residual, motion vector) contributes.

**Inputs you provide** (next cell):
- the re-encoded HMDB51 videos and index from `python dataloaders/prepare_hmdb51_dataset.py --download-path <dir>`;
- one or more **trained** checkpoints (`pytorch_model.bin.<epoch>` from `main_task_retrieval.py`).

Each entry in `VARIANTS` is a checkpoint plus the visual branches to build. Two kinds of ablation are supported:
- **Inference-time:** one full checkpoint, evaluated with branches dropped. The default `VARIANTS` list does this.
- **Trained:** a checkpoint trained with `--visual_branches ...`, evaluated with those same branches.

The results label them by checkpoint, so the two kinds are never mixed up.

This notebook is committed **without outputs**; no results exist until it is run on a trained checkpoint.

**Metrics.** Top-1/5/10 accuracy, mean per-class accuracy, and median and mean rank of the true class (ties count against it). They are reported on all videos and on the test videos of the three official splits; the *split mean* is the usual HMDB51 headline number. Videos that fail to decode give NaN embeddings and are excluded and counted, never silently scored.

In [ ]:
# ---------------- Configuration ----------------
HMDB51_ROOT = "/tmp/hmdb51"                       # --download-path given to prepare_hmdb51_dataset.py
VIDEOS_DIR = f"{HMDB51_ROOT}/mpeg4_videos"
INDEX_CSV = f"{HMDB51_ROOT}/hmdb51_index.csv"

CHECKPOINT = None                                  # e.g. "/storage/ckpts/full/pytorch_model.bin.4"
PRETRAINED_CLIP_NAME = "ViT-B/32"                  # architecture of the checkpoint

ALL = ["iframe", "residual", "mv"]
VARIANTS = [
    # name               checkpoint            visual branches built at inference
    {"name": "full",            "checkpoint": CHECKPOINT, "visual_branches": ALL},
    {"name": "no_mv",           "checkpoint": CHECKPOINT, "visual_branches": ["iframe", "residual"]},
    {"name": "no_residual",     "checkpoint": CHECKPOINT, "visual_branches": ["iframe", "mv"]},
    {"name": "no_iframe",       "checkpoint": CHECKPOINT, "visual_branches": ["residual", "mv"]},
    {"name": "iframe_only",     "checkpoint": CHECKPOINT, "visual_branches": ["iframe"]},
    {"name": "residual_only",   "checkpoint": CHECKPOINT, "visual_branches": ["residual"]},
    {"name": "mv_only",         "checkpoint": CHECKPOINT, "visual_branches": ["mv"]},
    # A model *trained* without a branch, e.g.:
    # {"name": "trained_no_mv", "checkpoint": "/storage/ckpts/no_mv/pytorch_model.bin.4",
    #  "visual_branches": ["iframe", "residual"]},
]
BASELINE = "full"                                  # the ablation table reports deltas against this variant

PROMPT_SETS = ["label", "phrase", "a person is {phrase}"]
REPORT_VARIANT, REPORT_PROMPT = "full", "a person is {phrase}"   # for per-class accuracy / confusion matrix
EVAL_SPLITS = [1, 2, 3]

DEVICE = "cuda"                                    # "cpu" works, slowly
BATCH_SIZE = 8
NUM_WORKERS = 4
# Same sampling as training (MSRVTT_Compressed_DataLoader defaults).
MAX_IFRAME_LENGTH, MAX_RESIDUAL_LENGTH, MAX_MV_LENGTH = 100, 100, 100
MAX_WORDS = 32

CACHE_DIR = f"{HMDB51_ROOT}/embedding_cache"       # per-variant video embeddings; delete to recompute
LIMIT_VIDEOS = None                                # e.g. 200 for a quick partial run (reported as such)

# Pipeline check ONLY: allow checkpoint=None, i.e. CLIP-initialised weights with a
# *random* motion-vector tower. Numbers from such a run are not results.
SMOKE_TEST_CLIP_INIT = False

In [ ]:
import os, sys, json, hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()   # compressed_domain/
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

import action_recognition as ar
from dataloaders.dataloader_hmdb51_compressed import dataloader_hmdb51
from dataloaders.prepare_hmdb51_dataset import HMDB51_CLASSES
from modules.tokenization_clip import SimpleTokenizer

assert len({v["name"] for v in VARIANTS}) == len(VARIANTS), "variant names must be unique"
missing = [v["name"] for v in VARIANTS if v["checkpoint"] is None]
if missing and not SMOKE_TEST_CLIP_INIT:
    raise ValueError(f"No checkpoint for variants {missing}. Set CHECKPOINT / VARIANTS to trained checkpoints "
                     "(or SMOKE_TEST_CLIP_INIT = True to only check that the pipeline runs).")
for v in VARIANTS:
    if v["checkpoint"] is not None and not os.path.isfile(v["checkpoint"]):
        raise FileNotFoundError(v["checkpoint"])
for path in (VIDEOS_DIR, INDEX_CSV):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} -- run dataloaders/prepare_hmdb51_dataset.py first")

PIPELINE_CHECK = bool(missing) or LIMIT_VIDEOS is not None
if missing:
    print("!" * 78)
    print("!!  SMOKE TEST: CLIP-initialised weights, random motion-vector tower.")
    print("!!  This only checks that the pipeline runs. The numbers below are NOT results.")
    print("!" * 78)
if LIMIT_VIDEOS is not None:
    print(f"Partial run: first {LIMIT_VIDEOS} videos only. Not comparable to full-dataset results.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    DEVICE = "cpu"
    print("CUDA unavailable, using CPU")

## Video embeddings
Each batch is decoded once and passed through every variant that has no cached embeddings. A cache entry is reused only if its checkpoint, branches, video list and sampling settings all match.

In [ ]:
loader, n_videos = dataloader_hmdb51(
    INDEX_CSV, VIDEOS_DIR, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    max_iframe_length=MAX_IFRAME_LENGTH, max_residual_length=MAX_RESIDUAL_LENGTH, max_mv_length=MAX_MV_LENGTH)
index = loader.dataset.index
if LIMIT_VIDEOS is not None:
    index = index.iloc[:LIMIT_VIDEOS]
    loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(loader.dataset, range(len(index))), batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, collate_fn=loader.collate_fn)
print(f"{len(index)} videos, {index['label'].nunique()} classes")

def cache_key(variant):
    ckpt = variant["checkpoint"]
    stamp = None if ckpt is None else [os.path.abspath(ckpt), os.path.getsize(ckpt), os.path.getmtime(ckpt)]
    key = {"checkpoint": stamp, "branches": sorted(variant["visual_branches"]), "clip": PRETRAINED_CLIP_NAME,
           "videos": hashlib.sha1("\n".join(index["video_id"]).encode()).hexdigest(),
           "lengths": [MAX_IFRAME_LENGTH, MAX_RESIDUAL_LENGTH, MAX_MV_LENGTH]}
    return json.dumps(key, sort_keys=True)

os.makedirs(CACHE_DIR, exist_ok=True)
video_emb, decode_ok = {}, None
for v in VARIANTS:
    path = Path(CACHE_DIR) / f"{v['name']}.npz"
    if path.exists():
        cached = np.load(path)
        if str(cached["key"]) == cache_key(v):
            video_emb[v["name"]] = cached["embeddings"]
            decode_ok = cached["decode_ok"]

todo = [v for v in VARIANTS if v["name"] not in video_emb]
print("cached:", [n for n in video_emb] or "none", "| to compute:", [v["name"] for v in todo] or "none")

models = {v["name"]: ar.load_variant(v["checkpoint"], v["visual_branches"], DEVICE, PRETRAINED_CLIP_NAME)
          for v in VARIANTS}
if todo:
    new_emb, meta = ar.extract_video_embeddings({v["name"]: models[v["name"]] for v in todo}, loader, DEVICE,
                                                progress=lambda it: tqdm(it, desc="videos"))
    assert (meta["index"] == np.arange(len(index))).all()
    decode_ok = meta["decode_ok"]
    for v in todo:
        video_emb[v["name"]] = new_emb[v["name"]]
        np.savez(Path(CACHE_DIR) / f"{v['name']}.npz", embeddings=new_emb[v["name"]], decode_ok=decode_ok,
                 key=cache_key(v))

labels = index["label_index"].to_numpy()
failed = index.loc[~decode_ok, "video_id"].tolist()
print(f"failed decodes: {len(failed)}", failed[:20])

## Text embeddings and scores
Text embeddings are computed per variant, because different checkpoints have different text towers.

In [ ]:
tokenizer = SimpleTokenizer()
scores = {}   # (variant, prompt set) -> (N_videos, 51) cosine scores
for v in VARIANTS:
    model = models[v["name"]]
    for prompt_set in PROMPT_SETS:
        ids, mask, segment = ar.tokenize_prompts(ar.class_prompts(prompt_set), tokenizer, MAX_WORDS)
        text = ar.encode_text(model, ids.to(DEVICE), mask.to(DEVICE), segment.to(DEVICE)).cpu().numpy()
        scores[v["name"], prompt_set] = ar.classify(video_emb[v["name"]], text)

eval_sets = {"all videos": np.ones(len(index), dtype=bool)}
for s in EVAL_SPLITS:
    eval_sets[f"split{s} test"] = (index[f"split{s}"] == 2).to_numpy()

rows = []
for v in VARIANTS:
    for prompt_set in PROMPT_SETS:
        for eval_name, subset in eval_sets.items():
            if not subset.any():
                continue
            m, _, _ = ar.action_recognition_metrics(scores[v["name"], prompt_set][subset], labels[subset],
                                                    num_classes=len(HMDB51_CLASSES))
            rows.append({"variant": v["name"], "checkpoint": v["checkpoint"] or "CLIP init (smoke test)",
                         "branches": "+".join(v["visual_branches"]), "prompts": prompt_set, "eval": eval_name, **m})
results = pd.DataFrame(rows)

split_mean = (results[results["eval"].str.startswith("split")]
              .groupby(["variant", "prompts"], sort=False)
              [["top1", "top5", "top10", "mean_class_acc", "median_rank", "mean_rank"]].mean().reset_index())
if PIPELINE_CHECK:
    print("PIPELINE CHECK -- not results")
split_mean.round(2)

### Full results
Every variant × prompt set × evaluation set.

In [ ]:
results.round(2)

## Ablation comparison
Top-1 accuracy, averaged over the official test splits, as a difference from the baseline variant. If no split test videos are present (for example in a partial run), all videos are used instead.

In [ ]:
table = split_mean if len(split_mean) else results[results["eval"] == "all videos"]
top1 = table.pivot(index="variant", columns="prompts", values="top1").loc[[v["name"] for v in VARIANTS], PROMPT_SETS]
ablation = top1.copy()
ablation.columns = [f"top1 · {c}" for c in ablation.columns]
for c in PROMPT_SETS:
    ablation[f"Δ vs {BASELINE} · {c}"] = top1[c] - top1.loc[BASELINE, c]
ablation.insert(0, "branches", [ "+".join(v["visual_branches"]) for v in VARIANTS])
ablation.round(2)

In [ ]:
# Grouped bars: top-1 per variant, one bar per prompt set (categorical slots 1-3).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(top1))
width = 0.8 / len(PROMPT_SETS)
for i, prompt_set in enumerate(PROMPT_SETS):
    bars = ax.bar(x + (i - (len(PROMPT_SETS) - 1) / 2) * width, top1[prompt_set], width * 0.92,
                  color=SERIES[i], label=prompt_set)
    if prompt_set == REPORT_PROMPT:
        ax.bar_label(bars, fmt="%.1f", fontsize=8, color=MUTED, padding=2)
ax.set_xticks(x, top1.index, rotation=20, ha="right", color=INK)
ax.set_ylabel("top-1 accuracy (%)", color=MUTED)
ax.set_ylim(0, max(5.0, float(np.nanmax(top1.to_numpy())) * 1.15))
ax.set_title("Zero-shot HMDB51 top-1 by variant" + ("  [PIPELINE CHECK, not results]" if PIPELINE_CHECK else ""),
             loc="left", color=INK)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, title="prompt set", ncols=len(PROMPT_SETS), loc="upper right")
plt.tight_layout()
plt.show()

## Per-class accuracy
The `REPORT_VARIANT` with `REPORT_PROMPT`, on all videos.

In [ ]:
metrics, per_class, confusion = ar.action_recognition_metrics(
    scores[REPORT_VARIANT, REPORT_PROMPT], labels, num_classes=len(HMDB51_CLASSES))
per_class_df = (pd.DataFrame({"class": HMDB51_CLASSES, "prompt": ar.class_prompts(REPORT_PROMPT),
                              "videos": confusion.sum(axis=1), "accuracy": per_class})
                .sort_values("accuracy", ascending=False).reset_index(drop=True))
print(f"{REPORT_VARIANT} / {REPORT_PROMPT}:", {k: round(v, 2) for k, v in metrics.items()})

shown = per_class_df.dropna(subset=["accuracy"]).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 0.22 * len(shown) + 1))
ax.barh(shown["class"], shown["accuracy"], color="#2a78d6", height=0.7)
ax.axvline(100 / len(HMDB51_CLASSES), color=MUTED, linewidth=1, linestyle="--")
ax.text(100 / len(HMDB51_CLASSES), -0.6, f" chance ({100 / len(HMDB51_CLASSES):.1f}%)", color=MUTED, fontsize=8, va="top")
ax.set_xlim(0, 100)
ax.set_xlabel("accuracy (%)", color=MUTED)
ax.tick_params(axis="y", labelsize=8)
ax.set_title(f"Per-class accuracy — {REPORT_VARIANT}, '{REPORT_PROMPT}'", loc="left", color=INK)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()
per_class_df.round(1)

## Confusion matrix
Rows are the true class and columns the predicted class, normalised per row. The table lists the most frequent confusions.

In [ ]:
with np.errstate(invalid="ignore", divide="ignore"):
    normalised = confusion / confusion.sum(axis=1, keepdims=True) * 100
blues = LinearSegmentedColormap.from_list("seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(normalised, cmap=blues, vmin=0, vmax=100)
ax.set_xticks(range(len(HMDB51_CLASSES)), HMDB51_CLASSES, rotation=90, fontsize=7)
ax.set_yticks(range(len(HMDB51_CLASSES)), HMDB51_CLASSES, fontsize=7)
ax.set_xlabel("predicted", color=MUTED)
ax.set_ylabel("true", color=MUTED)
ax.set_title(f"Confusion (% of true class) — {REPORT_VARIANT}, '{REPORT_PROMPT}'", loc="left", color=INK)
fig.colorbar(im, ax=ax, shrink=0.6, label="% of the true class's videos")
plt.tight_layout()
plt.show()

off = confusion.copy()
np.fill_diagonal(off, 0)
pairs = [(HMDB51_CLASSES[t], HMDB51_CLASSES[p], off[t, p], normalised[t, p])
         for t, p in zip(*np.unravel_index(np.argsort(off, axis=None)[::-1][:15], off.shape)) if off[t, p] > 0]
pd.DataFrame(pairs, columns=["true", "predicted", "videos", "% of true class"]).round(1)

## Failed decodes
These videos are excluded from every metric above.

In [ ]:
pd.DataFrame({"video_id": failed})